# 13 分布式训练调试手册

## 概述

分布式训练的调试比单卡训练困难得多——错误可能来自通信、数值、显存、死锁等多个方面，
且错误信息往往晦涩难懂。本笔记本作为**调试手册**，系统梳理常见错误类型、排查方法和工具链，
帮助你在遇到问题时快速定位和解决。

### 🧠 直觉理解：调试分布式训练像侦探破案

调试分布式训练就像侦探破案，需要系统的方法论：

1. **收集线索（日志）**：查看 NCCL 日志、PyTorch 错误栈、系统日志——就像收集现场证据
2. **缩小范围（隔离）**：用单卡 vs 多卡对比、减少 GPU 数量、关闭混合精度——就像排除嫌疑人
3. **验证假设（单卡对比）**：在单卡上复现问题、对比数值差异——就像在实验室验证推理

最关键的原则：**先让单卡跑通，再扩展到多卡**。90% 的分布式问题在单卡上就能发现。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

print("分布式训练调试手册 - 可视化分析")

## 1. 常见错误分类

分布式训练的错误可以分为以下几大类：

| 错误类型 | 典型表现 | 难度 | 频率 |
|----------|----------|------|------|
| NCCL 通信错误 | Timeout、Connection refused | ⭐⭐⭐ | 高 |
| 梯度 NaN | Loss 突然变 NaN | ⭐⭐ | 中 |
| OOM 显存不足 | CUDA out of memory | ⭐ | 高 |
| 死锁 | 程序挂起不动 | ⭐⭐⭐⭐ | 低 |
| 数值不一致 | 不同 rank 结果不同 | ⭐⭐⭐ | 中 |

In [ ]:
error_types = ['NCCL通信', '梯度NaN', 'OOM', '死锁', '数值不一致']
frequencies = [35, 20, 25, 10, 10]

fig, ax = plt.subplots(figsize=(8, 5))
ax.pie(frequencies, labels=error_types, autopct='%1.0f%%', startangle=90, 
       colors=['#e74c3c', '#f39c12', '#3498db', '#9b59b6', '#2ecc71'])
ax.set_title('分布式训练常见错误分布')
plt.tight_layout()
plt.show()

## 2. NCCL 错误排查

NCCL（NVIDIA Collective Communications Library）是多 GPU 通信的基础库，其错误是最常见的分布式训练问题。

### 常见 NCCL 错误

| 错误信息 | 原因 | 解决方法 |
|----------|------|----------|
| `NCCL error: unhandled system error` | NCCL 版本不兼容 | 更新 NCCL：`conda install nccl` |
| `NCCL error: timeout` | 某个 rank 卡住 | 检查负载均衡、设置 `NCCL_DEBUG=INFO` |
| `Connection refused` | 端口被占用或防火墙 | 更换端口、检查防火墙设置 |
| `No such file or directory` | 共享文件系统问题 | 检查 NFS 挂载、使用 `NCCL_SOCKET_IFNAME` |

### 关键环境变量

```bash
# 开启 NCCL 调试日志
export NCCL_DEBUG=INFO

# 指定网络接口（多网卡时重要）
export NCCL_SOCKET_IFNAME=eth0

# 设置超时时间（默认 30 分钟）
export NCCL_COMM_BLOCKING=1

# 禁用 IB，强制使用 TCP（调试用）
export NCCL_IB_DISABLE=1
```

## 3. 数值问题排查

### 梯度 NaN

**症状**：Loss 突然变为 NaN，通常在训练初期或中期出现。

**常见原因**：
1. **学习率过大**：梯度爆炸 → 参数溢出 → NaN
2. **混合精度溢出**：FP16 动态范围不足（最大 65504）
3. **除零错误**：RMSNorm/LayerNorm 的 eps 太小
4. **数据问题**：输入包含 NaN 或 Inf

**排查步骤**：
```python
# 1. 检查输入数据
assert not torch.isnan(input_ids).any()

# 2. 检查梯度
for name, param in model.named_parameters():
    if param.grad is not None and torch.isnan(param.grad).any():
        print(f"NaN gradient in: {name}")

# 3. 使用梯度裁剪
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

# 4. 检查 Loss 缩放（混合精度）
print(f"Loss scale: {scaler.get_scale()}")
```

### Loss 爆炸

**症状**：Loss 突然从正常值飙升到极大值。

**常见原因**：学习率过大、数据异常、warmup 不足。

**解决方法**：降低学习率、增加 warmup 步数、使用梯度裁剪。

## 4. 显存问题排查

OOM（Out of Memory）是分布式训练中最常见也最容易解决的问题。

### 显存组成分析

训练时 GPU 显存主要由以下部分组成：
- **模型参数**：与参数量成正比
- **梯度**：与参数量相同大小
- **优化器状态**：Adam 需要 2 倍参数量的状态（m 和 v）
- **激活值**：与 batch_size × seq_len 成正比
- **KV Cache**：推理时的额外开销

In [ ]:
# 模拟显存分析
components = ['模型参数', '梯度', '优化器状态', '激活值', 'KV Cache']
sizes_7b = [14, 14, 28, 8, 2]  # GB (fp32 optimizer)
sizes_70b = [140, 140, 280, 40, 10]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, sizes, title in zip(axes, [sizes_7b, sizes_70b], ['7B 模型', '70B 模型']):
    ax.bar(components, sizes, color=['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6'])
    ax.set_ylabel('显存 (GB)')
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=30)
plt.suptitle('模型训练显存组成分析')
plt.tight_layout()
plt.show()

In [ ]:
# OOM 解决方案对比
solutions = ['梯度累积', '混合精度', 'ZeRO-1', 'ZeRO-2', 'ZeRO-3', '激活重计算']
mem_reduction = [0.0, 0.5, 0.33, 0.5, 0.75, 0.6]  # 显存减少比例
compute_overhead = [0.0, 0.0, 0.05, 0.05, 0.1, 0.33]  # 计算开销增加

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(solutions))
width = 0.35

bars1 = ax.bar(x - width/2, mem_reduction, width, label='显存节省比例', color='#2ecc71')
bars2 = ax.bar(x + width/2, compute_overhead, width, label='计算开销增加', color='#e74c3c')

ax.set_xlabel('解决方案')
ax.set_ylabel('比例')
ax.set_title('OOM 解决方案：显存节省 vs 计算开销')
ax.set_xticks(x)
ax.set_xticklabels(solutions, rotation=30)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 5. 调试工具清单

### 基础工具

| 工具 | 用途 | 使用方式 |
|------|------|----------|
| `torch.distributed.launch` | 启动分布式训练 | `python -m torch.distributed.launch --nproc_per_node=8 train.py` |
| `NCCL_DEBUG=INFO` | 查看 NCCL 通信日志 | 环境变量 |
| `torch.profiler` | 性能分析 | 代码内集成 |
| `nvidia-smi` | GPU 状态监控 | 命令行 |

### 进阶工具

| 工具 | 用途 | 使用方式 |
|------|------|----------|
| `torch.autograd.detect_anomaly()` | 自动检测 NaN/Inf | 上下文管理器 |
| `torch.distributed.barrier()` | 同步点调试 | 代码内插入 |
| `torch.cuda.memory_summary()` | 显存详细报告 | 代码内调用 |
| `torch.utils.benchmark` | 性能基准测试 | 代码内集成 |

### 实用代码片段

```python
# 检测 NaN 梯度
with torch.autograd.detect_anomaly():
    loss.backward()

# 打印显存使用
print(torch.cuda.memory_summary())

# 检查 rank 间数值一致性
torch.distributed.all_reduce(tensor, op=torch.distributed.ReduceOp.SUM)
```

## 6. 通用排查流程

```
分布式训练出错
│
├── 第一步：单卡能否复现？
│   ├── 能 → 问题在模型/数据，先修单卡
│   └── 不能 → 问题在分布式，继续排查
│
├── 第二步：错误类型判断
│   ├── NCCL 错误 → 检查网络、端口、NCCL 版本
│   ├── NaN/Inf → 检查学习率、混合精度、数据
│   ├── OOM → 减 batch size、开混合精度、用 ZeRO
│   └── 死锁 → 检查 barrier、集合通信匹配
│
├── 第三步：最小复现
│   ├── 减少 GPU 数量（8→4→2→1）
│   ├── 减少模型大小
│   └── 使用合成数据
│
└── 第四步：逐步恢复
    ├── 单卡通过 → 2 卡 → 4 卡 → 8 卡
    ├── 每步验证数值一致性
    └── 出错时对比差异定位问题
```

In [ ]:
# 排查流程可视化
fig, ax = plt.subplots(figsize=(10, 8))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('分布式训练调试流程', fontsize=14, fontweight='bold')

steps = [
    (5, 9, '训练出错', '#e74c3c'),
    (5, 7.5, '单卡能否复现？', '#f39c12'),
    (2, 6, '修模型/数据', '#2ecc71'),
    (8, 6, '判断错误类型', '#3498db'),
    (5, 4.5, '最小复现', '#9b59b6'),
    (5, 3, '逐步恢复', '#1abc9c'),
    (5, 1.5, '问题解决 ✓', '#2ecc71'),
]

for x, y, text, color in steps:
    rect = plt.Rectangle((x-1.5, y-0.4), 3, 0.8, facecolor=color, alpha=0.2, edgecolor=color, linewidth=2, zorder=3)
    ax.add_patch(rect)
    ax.text(x, y, text, ha='center', va='center', fontsize=10, fontweight='bold', zorder=4)

# 箭头
arrows = [
    (5, 8.6, 5, 7.9),
    (3.5, 7.1, 2, 6.4),
    (6.5, 7.1, 8, 6.4),
    (8, 5.6, 5, 4.9),
    (5, 4.1, 5, 3.4),
    (5, 2.6, 5, 1.9),
]
for x1, y1, x2, y2 in arrows:
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))

# 标注
ax.text(2.5, 6.8, '能', fontsize=9, color='#2ecc71', fontweight='bold')
ax.text(7, 6.8, '不能', fontsize=9, color='#e74c3c', fontweight='bold')

plt.tight_layout()
plt.show()

## 7. 调试 Checklist

遇到分布式训练问题时，按以下清单逐项检查：

### 环境检查
- [ ] CUDA 版本与驱动兼容
- [ ] NCCL 版本与 PyTorch 匹配
- [ ] 网络接口配置正确（`NCCL_SOCKET_IFNAME`）
- [ ] 防火墙未阻拦通信端口

### 代码检查
- [ ] 所有 rank 执行相同的集合通信操作
- [ ] `torch.distributed.init_process_group()` 正确调用
- [ ] 数据加载使用 `DistributedSampler`
- [ ] 梯度裁剪在 `allreduce` 之后

### 数值检查
- [ ] 单卡 loss 与多卡 loss 一致
- [ ] 梯度未出现 NaN/Inf
- [ ] 混合精度 loss scale 正常
- [ ] 学习率 warmup 充分

### 显存检查
- [ ] `nvidia-smi` 确认显存使用
- [ ] batch size 在显存限制内
- [ ] 激活重计算已开启（如需要）
- [ ] ZeRO 分片配置正确

In [ ]:
# 调试时间分布统计（模拟数据）
debug_phases = ['环境配置', '单卡验证', '多卡调试', '数值排查', '性能优化']
time_pct = [15, 20, 30, 25, 10]
colors = ['#e74c3c', '#f39c12', '#3498db', '#9b59b6', '#2ecc71']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(debug_phases, time_pct, color=colors)
ax.set_xlabel('调试时间占比 (%)')
ax.set_title('分布式训练调试时间分布')

for bar, pct in zip(bars, time_pct):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2., f'{pct}%', va='center', fontsize=11)

ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 📝 练习题

1. **NCCL Timeout 排查**：你的 8 卡训练任务在启动后 5 分钟报 `NCCL error: timeout`，但单卡训练正常。请列出至少 3 种可能的原因和对应的排查方法。

2. **梯度 NaN 调试**：训练一个 7B 模型时，loss 在第 2000 步突然变为 NaN。你使用的是混合精度训练（FP16），学习率为 2e-4。请设计一个系统的排查方案，逐步缩小问题范围。

3. **OOM 优化方案**：训练 70B 模型时遇到 OOM，当前配置为 TP=8、batch_size=4、seq_len=4096、fp16。请列出至少 4 种减少显存占用的方法，并分析每种方法的计算开销。